# Operators: `QubitOperator`, `FermionOperator`, transforms, and grouping

qarp ships its own operator classes, backed by a packed binary-symplectic C++ core and
exposed with the **openfermion-compatible API** — same constructors, same `.terms`, same
arithmetic — without openfermion as a dependency:

- classes: `qarp.operators.QubitOperator` / `FermionOperator`
- free functions: `qarp.operators.functions` (`jordan_wigner`, `bravyi_kitaev`,
  `parity_transform`, `hermitian_conjugated`, `eigenspectrum`, …)
- dense/sparse matrices: `op.sparse_matrix()` — qarpx LSB, like every statevector
- measurement grouping: `qarp.operators` (`FullyCommuting`, `QubitWiseCommuting`, …)
- openfermion interop (optional): `qarp.operators.compat` (converters + openfermion's
  MSB `get_sparse_operator`)

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla

from qarp.operators import FermionOperator, QubitOperator
from qarp.operators.functions import (
    bravyi_kitaev,
    hermitian_conjugated,
    is_hermitian,
    jordan_wigner,
    parity_transform,
)

## 1. `QubitOperator` basics

Constructors follow openfermion exactly: a term string (`"X0 Z2"`), a factor tuple, or
nothing (zero operator). Coefficients can be any numeric type.

In [ ]:
h = (QubitOperator("X0 X1", 0.5)
     + QubitOperator("Y0 Y1", 0.5)
     + QubitOperator("Z0 Z1", 0.6)
     + QubitOperator("Z0", 0.3)
     + QubitOperator(((1, "Z"),), 0.3)       # tuple form, same as "Z1"
     + QubitOperator.identity(0.1))          # constant term

print(h)
print("qubits:", h.count_qubits(), "| terms:", len(h.terms), "| hermitian:", is_hermitian(h))

Full operator algebra: products expand through the Pauli group (`X·Y = iZ` per qubit),
scalars work from both sides, and `**` powers the whole sum. `.terms` is a **read-only
view** (assign a whole dict to replace it); `coefficient(term)` is a fast single-term
lookup (a qarpx extra).

In [ ]:
x0, y0 = QubitOperator("X0"), QubitOperator("Y0")
print("X0 * Y0             =", x0 * y0)                 # iZ0
print("commutator [X0,Y0]  =", x0 * y0 - y0 * x0)       # 2iZ0

square = (x0 + y0) ** 2                                  # XY + YX cancels...
square.compress()                                        # ...compress prunes the 0j term
print("(X0 + Y0)**2        =", square)                   # 2·identity

# coefficient() takes a .terms-style key (fast single-term lookup, qarpx extra):
print("h coefficient of Z0Z1:", h.coefficient(((0, "Z"), (1, "Z"))))
print("h dagger == h:", hermitian_conjugated(h) == h)

tiny = h + QubitOperator("X0", 1e-14)
tiny.compress()                                          # drop numeric noise
print("compress removed the 1e-14 term:", tiny.isclose(h))

## 2. `FermionOperator` and the fermion→qubit transforms

Ladder terms use openfermion's grammar: `"2^ 0"` is $a_2^\dagger a_0$. The three
standard encodings are available as free functions — scalars or whole lists in one call.

In [ ]:
# Two-site Hubbard-style Hamiltonian: hopping + on-site interaction.
hop = FermionOperator("0^ 1", -1.0) + FermionOperator("1^ 0", -1.0)
interaction = FermionOperator("0^ 0 1^ 1", 2.0)
h_f = hop + interaction
print(h_f)

h_jw = jordan_wigner(h_f)
h_bk = bravyi_kitaev(h_f)
h_parity = parity_transform(h_f, n_qubits=2)
print("\nJordan-Wigner:\n", h_jw, sep="")
print("hermitian after transform:", is_hermitian(h_jw))

The encodings differ term-by-term but are **spectrally identical** — same physics, and a
convenient endianness-free check:

In [ ]:
def spectrum(op):
    return np.sort(np.linalg.eigvalsh(op.sparse_matrix(2).toarray()))

for name, op in [("jordan_wigner", h_jw), ("bravyi_kitaev", h_bk), ("parity", h_parity)]:
    print(f"{name:<15}", np.round(spectrum(op), 10))

## 3. Sparse matrices

`op.sparse_matrix(n_qubits=None)` assembles a `scipy.sparse` matrix from a C++ COO kernel in
the **qarpx LSB convention: qubit `q` is bit `q`** of the state index — the same convention
as every qarpx statevector, sampled bitstring and block unitary, so matrices contract with
circuit output directly.  There is one convention in qarp.

> openfermion (and cirq / pennylane) use the opposite, MSB layout (qubit 0 = most
> significant bit).  That layout is available only for interop as
> `qarp.operators.compat.get_sparse_operator`; crossing an external MSB boundary uses the
> bit-reversal helpers in `qarp.endianness`.  Spectra never need any of this.

In [ ]:
h_sparse = h_jw.sparse_matrix()
ground = spla.eigsh(h_sparse, k=1, which="SA")[0][0]
print(f"{h_sparse.shape} sparse matrix, ground-state energy: {ground:.6f}")

# FermionOperator has its own direct kernel too (Jordan-Wigner, same LSB convention):
print("direct fermion kernel matches JW:",
      np.allclose(h_f.sparse_matrix(2).toarray(), h_sparse.toarray()))

## 4. Operators drive the algorithm layer

Every consumer takes these classes directly: `TrotterBlock(operator=...)` exponentiates
them, `PauliAveraging` measures them (engines route the operator to the simulator through
a one-crossing observable fast path). Here: evolve under `h`, then measure `h` exactly.

In [ ]:
from qarp import EXACT
from qarp.algorithms import PauliAveraging
from qarp.blocks import SimpleBlock, TrotterBlock
from qarp.engines import QarpEngine

state = SimpleBlock(2, name="prep")
state.x(0)                       # |01> in qubit order (q0=1, q1=0)
state.build()

evolution = TrotterBlock(operator=h, n_qubits=2, steps=4, time=0.7, order=2).build()

from qarp.blocks import CompositeBlock
ket = CompositeBlock([state, evolution], n_qubits=2)

measure = PauliAveraging(operator=h, ket=ket)
engine = QarpEngine(n_shots=EXACT)
engine.build([measure])
energy = engine.run()[0]

# Energy is conserved under evolution generated by h itself:
ref = PauliAveraging(operator=h, ket=state)
engine.build([ref])
print(f"<psi(0)|H|psi(0)> = {engine.run()[0]:.8f}")
print(f"<psi(t)|H|psi(t)> = {energy:.8f}   (conserved under exp(-iHt))")

## 5. Measurement grouping

`PauliAveraging` measures one circuit per **group** of simultaneously-measurable terms.
The partition is a pluggable `GroupingStrategy`; fewer groups = fewer circuits:

- `FullyCommuting` — general commutation, diagonalised by a Clifford (the
  `PauliAveraging` default)
- `QubitWiseCommuting` — the weaker qubit-wise criterion (no Clifford needed)
- `NoGrouping` — one circuit per term

In [ ]:
from qarp.operators import FullyCommuting, NoGrouping, QubitWiseCommuting

# A GroupingStrategy is a pure index partitioner over {qubit: letter} dicts.
terms = [dict(term) for term in h.terms if term]   # skip the identity constant
print("terms:", terms, "\n")
for strategy in (NoGrouping(), QubitWiseCommuting(), FullyCommuting()):
    groups = strategy.group(terms, n_qubits=2)
    print(f"{type(strategy).__name__:<20} {len(groups)} measurement circuits  "
          f"(term indices: {groups})")

## 6. Interop and extras

- **openfermion bridge** (optional dependency): `qarp.operators.compat.to_openfermion` /
  `from_openfermion` convert term-for-term.
- **Symbolic coefficients**: with the `QARP_WITH_SYMENGINE` build flag, coefficients can
  be sympy expressions (`substitute()` / `free_symbols()` appear on the classes); the
  default numeric build raises on symbolic input.
- **Pickle / deepcopy** work exactly like openfermion's classes.

In [ ]:
try:
    from qarp.operators.compat import from_openfermion, to_openfermion

    of_op = to_openfermion(h)
    back = from_openfermion(of_op)
    print("openfermion round-trip exact:", back == h, "|", type(of_op).__module__)
except ImportError as err:
    print("openfermion not installed - compat bridge unavailable:", err)

import copy, pickle
print("pickle round-trip exact:", pickle.loads(pickle.dumps(h)) == h)
print("deepcopy is independent:", copy.deepcopy(h) == h)